# 汎用感情認識ノートブック

生WAVファイルから emotion2vec で特徴量を抽出し、学習（VA回帰 → 感情分類）をLeave-One-Out交差検証で実行する汎用ノートブック。

**必要なCSV形式:**
```
file_path,session,label[,valence,arousal]
audio/sample001.wav,Session1,angry,2.5,3.2
```
- `valence`/`arousal` 列がある場合: Stage 1（VA回帰）→ Stage 2（感情分類）の2段階学習
- ない場合: Stage 2のみ実行

In [1]:
# Cell 1: 依存ライブラリのインストール
!pip install funasr modelscope soundfile torchaudio tqdm

In [2]:
# Cell 2: CONFIG — ユーザーが編集する唯一の場所
CSV_PATH    = "C:\\Users\\RD004\\Documents\\lab\\data\\iemocap\\IEMOCAP正解ラベル.csv"       # データCSVのパス
AUDIO_DIR   = "data/audio/"           # WAVのベースディレクトリ（file_pathが相対パスの場合）
CACHE_DIR   = "cache/"                # 特徴量キャッシュ保存先
CLASS_NAMES = ["angry", "happy", "neutral", "sad"]  # 感情クラス名（CSVのlabel列の値と一致させる）

BATCH_SIZE      = 32
STAGE1_EPOCHS   = 30    # VAラベルなし時は無視
STAGE2_EPOCHS   = 30
STAGE1_LR       = 1e-3
STAGE2_LR_FNN   = 1e-4
STAGE2_LR_CLS   = 1e-3

SEED = 42

In [3]:
# Cell 3: インポート
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

使用デバイス: cpu


In [4]:
# Cell 4: emotion2vec モデルのロード
from funasr import AutoModel

emotion2vec_model = AutoModel(model="iic/emotion2vec_base")
print("emotion2vec_base ロード完了")

Notice: ffmpeg is not installed. torchaudio is used to load audio
If you want to use ffmpeg backend to load audio, please install it by:
	sudo apt install ffmpeg # ubuntu
	# brew install ffmpeg # mac
funasr version: 1.3.1.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.1


emotion2vec_base ロード完了


In [5]:
# Cell 5: CSV読み込み・検証
df = pd.read_csv(CSV_PATH)

# 必須列チェック
required_cols = {"file_path", "session", "label"}
missing_cols = required_cols - set(df.columns)
assert not missing_cols, f"CSVに必須列が不足: {missing_cols}"

# VAラベルの有無を確認
has_va = "valence" in df.columns and "arousal" in df.columns
print(f"VAラベル: {'あり（Stage 1 実施）' if has_va else 'なし（Stage 1 スキップ）'}")

# ラベル整合性チェック
unknown_labels = set(df["label"].unique()) - set(CLASS_NAMES)
assert not unknown_labels, f"CLASS_NAMESに含まれない未知ラベル: {unknown_labels}"

# ラベルを整数インデックスに変換
label2idx = {name: i for i, name in enumerate(CLASS_NAMES)}
df["label_idx"] = df["label"].map(label2idx)

# VAラベルを[-1,1]に正規化（元が[1,5]スケールの場合）
if has_va:
    # 値の範囲チェック
    va_min = min(df["valence"].min(), df["arousal"].min())
    va_max = max(df["valence"].max(), df["arousal"].max())
    if va_min >= -1.0 and va_max <= 1.0:
        print(f"VAラベルはすでに[-1,1]範囲内 (min={va_min:.2f}, max={va_max:.2f})")
    else:
        # [min, max] -> [-1, 1] に線形正規化
        for col in ["valence", "arousal"]:
            col_min, col_max = df[col].min(), df[col].max()
            df[col] = 2.0 * (df[col] - col_min) / (col_max - col_min + 1e-8) - 1.0
        print(f"VAラベルを[-1,1]に正規化 (元の範囲: [{va_min:.2f}, {va_max:.2f}])")

sessions = sorted(df["session"].unique())
print(f"\nサンプル数: {len(df)}")
print(f"セッション数: {len(sessions)} ({sessions})")
print(f"クラス分布:\n{df['label'].value_counts().to_string()}")

AssertionError: CSVに必須列が不足: {'label', 'session', 'file_path'}

In [ ]:
# Cell 6: 特徴量抽出（キャッシュ付き）
# 各発話の特徴量を {CACHE_DIR}/{row_index}.npy に保存
# 既存キャッシュはスキップ

def extract_and_cache_features(df, audio_dir, cache_dir, model):
    audio_dir = Path(audio_dir)
    cache_dir = Path(cache_dir)
    
    need_extract = [
        i for i in range(len(df))
        if not (cache_dir / f"{i}.npy").exists()
    ]
    
    if not need_extract:
        print("全キャッシュ済み、特徴量抽出をスキップ")
        return
    
    print(f"{len(need_extract)} 件を抽出（全 {len(df)} 件中）")
    
    for i in tqdm(need_extract, desc="特徴量抽出"):
        row = df.iloc[i]
        file_path = Path(row["file_path"])
        if not file_path.is_absolute():
            file_path = audio_dir / file_path
        
        assert file_path.exists(), f"音声ファイルが見つからない: {file_path}"
        
        # emotion2vec で特徴量を抽出 (T, 768)
        result = model.generate(
            str(file_path),
            output_dir=None,
            granularity="frame",
            extract_embedding=True,
        )
        feats = result[0]["feats"]  # numpy (T, 768)
        np.save(str(cache_dir / f"{i}.npy"), feats.astype(np.float32))

extract_and_cache_features(df, AUDIO_DIR, CACHE_DIR, emotion2vec_model)
print("特徴量抽出完了")

In [ ]:
# Cell 7: モデル定義（インライン）

class AttentionPooling(nn.Module):
    """スコアアテンションによる可変長フレーム列の発話レベル圧縮。"""

    def __init__(self, input_dim: int):
        super().__init__()
        self.score = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D), padding_mask: (B, T) True=パディング
        scores = self.score(x).squeeze(-1)  # (B, T)
        scores = scores.masked_fill(padding_mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)  # (B, T)
        return (weights.unsqueeze(-1) * x).sum(dim=1)  # (B, D)


class VADDecoder(nn.Module):
    """AttentionPooling + FNN で発話表現を3次元VAD空間に写像。出力はTanh で[-1,1]。"""

    def __init__(self, input_dim: int = 768, hidden_dim: int = 256, vad_dim: int = 3):
        super().__init__()
        self.pool = AttentionPooling(input_dim)
        self.fnn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vad_dim),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor) -> torch.Tensor:
        pooled = self.pool(x, padding_mask)  # (B, D)
        return self.fnn(pooled)              # (B, vad_dim)


class EmotionClassifier(nn.Module):
    """VADDecoder の出力を線形分類器でカテゴリ感情に変換する2段階モデル全体。"""

    def __init__(
        self,
        input_dim: int = 768,
        hidden_dim: int = 256,
        vad_dim: int = 3,
        num_classes: int = 4,
    ):
        super().__init__()
        self.vad_decoder = VADDecoder(input_dim, hidden_dim, vad_dim)
        self.classifier = nn.Linear(vad_dim, num_classes)

    def forward(self, x, padding_mask):
        vad = self.vad_decoder(x, padding_mask)
        logits = self.classifier(vad)
        return vad, logits


print("モデル定義完了")

In [ ]:
# Cell 8: データセット定義（インライン）

class SpeechDataset(Dataset):
    """DataFrameとキャッシュディレクトリから特徴量・ラベルを読み込むDataset。"""

    def __init__(self, sub_df: pd.DataFrame, cache_dir: str, has_va: bool = False):
        self.df = sub_df.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.has_va = has_va
        # 元のDataFrame行インデックスを保持（キャッシュファイル名に使用）
        self.original_indices = sub_df.index.tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        orig_idx = self.original_indices[i]
        feats = np.load(str(self.cache_dir / f"{orig_idx}.npy"))  # (T, 768)
        feats = torch.from_numpy(feats).float()

        row = self.df.iloc[i]
        item = {"feats": feats, "label": int(row["label_idx"])}

        if self.has_va:
            item["va"] = torch.tensor(
                [float(row["valence"]), float(row["arousal"])], dtype=torch.float32
            )
        return item


def collate_fn(samples):
    """可変長フレームをゼロパディングしてバッチ化する。"""
    if not samples:
        return {}

    feats_list = [s["feats"] for s in samples]
    sizes = [f.shape[0] for f in feats_list]
    T_max = max(sizes)
    feat_dim = feats_list[0].shape[1]

    collated = torch.zeros(len(samples), T_max, feat_dim)
    padding_mask = torch.zeros(len(samples), T_max, dtype=torch.bool)

    for i, (feat, size) in enumerate(zip(feats_list, sizes)):
        collated[i, :size] = feat
        padding_mask[i, size:] = True

    batch = {
        "feats": collated,
        "padding_mask": padding_mask,
        "labels": torch.tensor([s["label"] for s in samples], dtype=torch.long),
    }

    if "va" in samples[0]:
        batch["va_labels"] = torch.stack([s["va"] for s in samples])  # (B, 2)

    return batch


print("データセット定義完了")

In [ ]:
# Cell 9: 損失関数・学習/評価関数（インライン）

def ccc_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred_mean = pred.mean()
    target_mean = target.mean()
    pred_var = pred.var(unbiased=False)
    target_var = target.var(unbiased=False)
    covariance = ((pred - pred_mean) * (target - target_mean)).mean()
    ccc = (2 * covariance) / (
        pred_var + target_var + (pred_mean - target_mean) ** 2 + 1e-8
    )
    return 1.0 - ccc


def stage1_loss(vad_pred: torch.Tensor, va_target: torch.Tensor) -> torch.Tensor:
    """Valence と Arousal の CCC損失の和。"""
    loss_v = ccc_loss(vad_pred[:, 0], va_target[:, 0])
    loss_a = ccc_loss(vad_pred[:, 1], va_target[:, 1])
    return loss_v + loss_a


def train_stage1(model, loader, optimizer, device):
    model.vad_decoder.train()
    model.classifier.eval()
    total_loss = 0.0
    for batch in loader:
        if "va_labels" not in batch:
            continue
        feats = batch["feats"].to(device)
        padding_mask = batch["padding_mask"].to(device)
        va_labels = batch["va_labels"].to(device)
        optimizer.zero_grad()
        vad, _ = model(feats, padding_mask)
        loss = stage1_loss(vad, va_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss


@torch.no_grad()
def eval_stage1(model, loader, device) -> float:
    model.eval()
    total_loss, n = 0.0, 0
    for batch in loader:
        if "va_labels" not in batch:
            continue
        feats = batch["feats"].to(device)
        padding_mask = batch["padding_mask"].to(device)
        va_labels = batch["va_labels"].to(device)
        vad, _ = model(feats, padding_mask)
        total_loss += stage1_loss(vad, va_labels).item()
        n += 1
    return total_loss / max(n, 1)


def train_stage2(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        feats = batch["feats"].to(device)
        padding_mask = batch["padding_mask"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        _, logits = model(feats, padding_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss


@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    """WA / UA / weighted-F1 (%) と混同行列を返す。"""
    model.eval()
    correct = total = 0
    uw_correct = [0] * num_classes
    uw_total   = [0] * num_classes
    tp = [0] * num_classes
    fp = [0] * num_classes
    fn = [0] * num_classes
    conf_matrix = np.zeros((num_classes, num_classes), dtype=int)
    all_vad = []
    all_labels = []

    for batch in loader:
        feats = batch["feats"].to(device)
        padding_mask = batch["padding_mask"].to(device)
        labels = batch["labels"].to(device)
        vad, logits = model(feats, padding_mask)
        predicted = logits.argmax(dim=1)

        total   += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_vad.append(vad.cpu().numpy())
        all_labels.extend(labels.cpu().tolist())

        for c, p in zip(labels.cpu().tolist(), predicted.cpu().tolist()):
            uw_total[c]   += 1
            conf_matrix[c, p] += 1
            if p == c:
                uw_correct[c] += 1
                tp[c] += 1
            else:
                fp[p] += 1
                fn[c] += 1

    wa  = correct / total * 100
    ua  = sum(uw_correct[i] / max(uw_total[i], 1) for i in range(num_classes)) / num_classes * 100
    wf1 = _weighted_f1(tp, fp, fn, uw_total) * 100
    all_vad = np.concatenate(all_vad, axis=0) if all_vad else None
    return wa, ua, wf1, conf_matrix, all_vad, all_labels


def _weighted_f1(tp, fp, fn, totals):
    f1s = []
    for i in range(len(tp)):
        prec = tp[i] / max(tp[i] + fp[i], 1)
        rec  = tp[i] / max(tp[i] + fn[i], 1)
        f1s.append(2 * prec * rec / max(prec + rec, 1e-8))
    return sum(f1s[i] * totals[i] for i in range(len(tp))) / max(sum(totals), 1)


print("損失関数・学習/評価関数定義完了")

In [ ]:
# Cell 10: Leave-One-Out 交差検証ループ

num_classes = len(CLASS_NAMES)
results = []  # (session, wa, ua, f1, conf_matrix, vad_preds, true_labels)
all_conf = np.zeros((num_classes, num_classes), dtype=int)
all_vad_preds = []
all_true_labels = []

for test_session in sessions:
    print(f"\n===== テストセッション: {test_session} =====")

    test_df  = df[df["session"] == test_session]
    train_df = df[df["session"] != test_session]

    train_ds = SpeechDataset(train_df, CACHE_DIR, has_va)
    test_ds  = SpeechDataset(test_df,  CACHE_DIR, has_va)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0
    )
    test_loader  = DataLoader(
        test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0
    )

    model = EmotionClassifier(
        input_dim=768, hidden_dim=256, vad_dim=3, num_classes=num_classes
    ).to(device)

    # ---------- Stage 1 (VAラベルあり時のみ) ----------
    if has_va:
        opt1 = optim.Adam(model.vad_decoder.parameters(), lr=STAGE1_LR)
        best_s1_loss = float("inf")
        best_s1_state = None

        s1_bar = tqdm(range(STAGE1_EPOCHS), desc="Stage1", leave=False)
        for epoch in s1_bar:
            train_loss = train_stage1(model, train_loader, opt1, device)
            val_loss   = eval_stage1(model, test_loader, device)
            s1_bar.set_postfix(train=f"{train_loss/max(len(train_loader),1):.4f}", val=f"{val_loss:.4f}")
            if val_loss < best_s1_loss:
                best_s1_loss = val_loss
                best_s1_state = {k: v.clone() for k, v in model.state_dict().items()}

        if best_s1_state is not None:
            model.load_state_dict(best_s1_state)
        print(f"  Stage 1 完了 | best val CCC loss: {best_s1_loss:.4f}")

    # ---------- Stage 2 ----------
    opt2 = optim.Adam([
        {"params": model.vad_decoder.parameters(), "lr": STAGE2_LR_FNN},
        {"params": model.classifier.parameters(), "lr": STAGE2_LR_CLS},
    ])
    criterion = nn.CrossEntropyLoss()
    best_s2_wa = 0.0
    best_s2_state = None

    s2_bar = tqdm(range(STAGE2_EPOCHS), desc="Stage2", leave=False)
    for epoch in s2_bar:
        train_loss = train_stage2(model, train_loader, opt2, criterion, device)
        val_wa, val_ua, val_f1, _, _, _ = evaluate(model, test_loader, device, num_classes)
        s2_bar.set_postfix(
            loss=f"{train_loss/max(len(train_loader),1):.4f}",
            WA=f"{val_wa:.1f}%",
            UA=f"{val_ua:.1f}%",
        )
        if val_wa > best_s2_wa:
            best_s2_wa = val_wa
            best_s2_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_s2_state is not None:
        model.load_state_dict(best_s2_state)

    # ---------- テスト評価 ----------
    test_wa, test_ua, test_f1, conf, vad_preds, true_lbls = evaluate(
        model, test_loader, device, num_classes
    )
    print(f"  Test WA={test_wa:.2f}%  UA={test_ua:.2f}%  F1={test_f1:.2f}%")

    results.append((test_session, test_wa, test_ua, test_f1, conf))
    all_conf += conf
    if vad_preds is not None:
        all_vad_preds.append(vad_preds)
    all_true_labels.extend(true_lbls)

# 平均スコア
avg_wa  = np.mean([r[1] for r in results])
avg_ua  = np.mean([r[2] for r in results])
avg_f1  = np.mean([r[3] for r in results])
print(f"\n{'='*50}")
print(f"平均 WA={avg_wa:.2f}%  UA={avg_ua:.2f}%  F1={avg_f1:.2f}%")
print(f"{'='*50}")

In [ ]:
# Cell 11: 結果集計・可視化

# --- 11-1: セッション別スコア表 ---
score_df = pd.DataFrame(
    [(r[0], f"{r[1]:.2f}", f"{r[2]:.2f}", f"{r[3]:.2f}") for r in results],
    columns=["Session", "WA(%)", "UA(%)", "F1(%)"],
)
score_df.loc[len(score_df)] = ["平均", f"{avg_wa:.2f}", f"{avg_ua:.2f}", f"{avg_f1:.2f}"]
print(score_df.to_string(index=False))

# --- 11-2: 全体の混同行列 ---
fig, axes = plt.subplots(1, 2 if has_va else 1, figsize=(12 if has_va else 6, 5))
if not has_va:
    axes = [axes]

ax_conf = axes[0]
im = ax_conf.imshow(all_conf, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax_conf)
ax_conf.set_xticks(range(num_classes))
ax_conf.set_yticks(range(num_classes))
ax_conf.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax_conf.set_yticklabels(CLASS_NAMES)
ax_conf.set_xlabel("予測")
ax_conf.set_ylabel("正解")
ax_conf.set_title("混同行列（全セッション合計）")
for i in range(num_classes):
    for j in range(num_classes):
        ax_conf.text(j, i, str(all_conf[i, j]), ha="center", va="center",
                     color="white" if all_conf[i, j] > all_conf.max() * 0.5 else "black")

# --- 11-3: VAD散布図（has_va=True の場合のみ） ---
if has_va and all_vad_preds:
    ax_vad = axes[1]
    vad_all = np.concatenate(all_vad_preds, axis=0)  # (N, 3)
    colors = plt.cm.tab10(np.linspace(0, 0.9, num_classes))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = np.array(all_true_labels) == cls_idx
        ax_vad.scatter(
            vad_all[mask, 0], vad_all[mask, 1],
            c=[colors[cls_idx]], label=cls_name, alpha=0.5, s=20,
        )
    ax_vad.set_xlabel("Valence")
    ax_vad.set_ylabel("Arousal")
    ax_vad.set_title("VAD空間の散布図（全セッション）")
    ax_vad.legend()
    ax_vad.set_xlim(-1.1, 1.1)
    ax_vad.set_ylim(-1.1, 1.1)
    ax_vad.axhline(0, color="gray", linewidth=0.5)
    ax_vad.axvline(0, color="gray", linewidth=0.5)

plt.tight_layout()
plt.savefig(Path(CACHE_DIR) / "results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"図を保存: {Path(CACHE_DIR) / 'results.png'}")